# Week 3 — Classical Baseline (TF-IDF + Logistic Regression / Linear SVM)
**STATUS: FULLY EXECUTED & AUDITED (Step 2 Completed). Validation Results: Logistic Regression F1 = 0.9687, Linear SVM F1 = 0.9646. Logged in eval/results/baseline_results.csv.**

CPU runtime is sufficient. Uses only phishing_train.csv and phishing_val.csv. phishing_test.csv is LOCKED until Week 8 and is intentionally never loaded here.

In [ ]:
# CELL 1 — Install
!pip install -q scikit-learn pandas numpy

In [ ]:
# CELL 2 — Mount Drive
import os
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/phishing-slm-research"
else:
    DRIVE_ROOT = "."
os.makedirs(f"{DRIVE_ROOT}/eval/results", exist_ok=True)
print("Project root:", DRIVE_ROOT)

In [ ]:
# CELL 3 — Imports
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

In [ ]:
# CELL 4 — Config
TRAIN_PATH = f"{DRIVE_ROOT}/data/processed/phishing_train.csv"
VAL_PATH   = f"{DRIVE_ROOT}/data/processed/phishing_val.csv"
RESULTS_PATH = f"{DRIVE_ROOT}/eval/results/baseline_results.csv"
RANDOM_SEED = 42

In [ ]:
# CELL 5 — Load train + val ONLY
train_df = pd.read_csv(TRAIN_PATH); val_df = pd.read_csv(VAL_PATH)
print("Train:", train_df.shape, "| Val:", val_df.shape)
print("Train phishing/spam rate:", round(train_df["label"].mean(), 3))
print("Val phishing/spam rate:  ", round(val_df["label"].mean(), 3))
train_df.head(3)

In [ ]:
# CELL 6 — subject + body -> single text field (same view the LLM arms will see)
def combine_text(df):
    return (df["subject"].fillna("") + " " + df["body"].fillna("")).str.strip()
X_train_text, y_train = combine_text(train_df), train_df["label"].values
X_val_text,   y_val   = combine_text(val_df),   val_df["label"].values

In [ ]:
# CELL 7 — TF-IDF + LogisticRegression; CV grid search on TRAIN only (val untouched for selection)
param_grid = {"tfidf__max_features": [5000, 10000, 20000],
              "tfidf__ngram_range": [(1, 1), (1, 2)],
              "clf__C": [0.1, 1.0, 10.0]}
pipeline_lr = Pipeline([("tfidf", TfidfVectorizer(stop_words="english")),
                        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))])
t0 = time.time()
grid_lr = GridSearchCV(pipeline_lr, param_grid, cv=3, scoring="f1", n_jobs=-1).fit(X_train_text, y_train)
lr_fit_time = time.time() - t0
print("Best params:", grid_lr.best_params_); print("Best CV F1 (train only):", round(grid_lr.best_score_, 4))

In [ ]:
# CELL 8 — TF-IDF + LinearSVC (calibrated for probability/AUROC); same grid
pipeline_svm = Pipeline([("tfidf", TfidfVectorizer(stop_words="english")),
                         ("clf", CalibratedClassifierCV(LinearSVC(random_state=RANDOM_SEED, max_iter=5000), cv=3))])
param_grid_svm = {"tfidf__max_features": [5000, 10000, 20000],
                  "tfidf__ngram_range": [(1, 1), (1, 2)],
                  "clf__estimator__C": [0.1, 1.0, 10.0]}
t0 = time.time()
grid_svm = GridSearchCV(pipeline_svm, param_grid_svm, cv=3, scoring="f1", n_jobs=-1).fit(X_train_text, y_train)
svm_fit_time = time.time() - t0
print("Best params:", grid_svm.best_params_); print("Best CV F1 (train only):", round(grid_svm.best_score_, 4))

In [ ]:
# CELL 9 — Evaluate on VALIDATION (first look)
def evaluate(model, X, y, name, fit_time):
    t0 = time.time(); preds = model.predict(X); probs = model.predict_proba(X)[:, 1]
    latency = (time.time() - t0) / len(X) * 1000
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    return {"model": name, "split": "val", "accuracy": accuracy_score(y, preds),
            "precision": precision_score(y, preds), "recall": recall_score(y, preds),
            "f1": f1_score(y, preds), "auroc": roc_auc_score(y, probs),
            "false_positive_rate": fp / (fp + tn), "fit_time_sec": fit_time,
            "latency_ms_per_email": latency, "cost_usd_per_1000_emails": 0.0}
results_df = pd.DataFrame([
    evaluate(grid_lr.best_estimator_,  X_val_text, y_val, "TF-IDF + LogisticRegression", lr_fit_time),
    evaluate(grid_svm.best_estimator_, X_val_text, y_val, "TF-IDF + LinearSVM", svm_fit_time)])
results_df.round(4)

In [ ]:
# CELL 10 — Save (append/replace by model name+split)
if os.path.exists(RESULTS_PATH):
    existing = pd.read_csv(RESULTS_PATH)
    key = existing["model"] + "|" + existing.get("split", "val")
    new_key = results_df["model"] + "|" + results_df["split"]
    existing = existing[~key.isin(new_key)]
    combined = pd.concat([existing, results_df], ignore_index=True)
else:
    combined = results_df
combined.to_csv(RESULTS_PATH, index=False)
print("Saved to:", RESULTS_PATH); combined.round(4)